In [62]:
#Import necessary libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

#Import the dataset
df = pd.read_csv("../data/SDSS_DR18.csv")

In [63]:
#Show basic info about the dataset variables
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 43 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   objid        100000 non-null  float64
 1   specobjid    100000 non-null  float64
 2   ra           100000 non-null  float64
 3   dec          100000 non-null  float64
 4   u            100000 non-null  float64
 5   g            100000 non-null  float64
 6   r            100000 non-null  float64
 7   i            100000 non-null  float64
 8   z            100000 non-null  float64
 9   run          100000 non-null  int64  
 10  rerun        100000 non-null  int64  
 11  camcol       100000 non-null  int64  
 12  field        100000 non-null  int64  
 13  plate        100000 non-null  int64  
 14  mjd          100000 non-null  int64  
 15  fiberid      100000 non-null  int64  
 16  petroRad_u   100000 non-null  float64
 17  petroRad_g   100000 non-null  float64
 18  petroRad_i   100000 non-n

In [ ]:
#Drop unnecessary columns that are not useful for classification
#Redshift dropped due to leaking information about the class labels, as it is closely related to the distance and type of celestial objects, which can directly influence the classification task. Including redshift could lead to overfitting and poor generalization on unseen data, as the model might rely too heavily on this feature rather than learning the underlying patterns in the other features.
drop_cols = [
    'objid',
    'specobjid',
    'run',
    'rerun',
    'camcol',
    'field',
    'plate',
    'mjd',
    'fiberid',
    'redshift',
    'ra',
    'dec'
]

#Check which columns from drop_cols actually exist in the DataFrame before dropping
existing_drop_cols = [col for col in drop_cols if col in df.columns]
#Print the remaining columns that will be dropped
print('Dropping columns:', existing_drop_cols)

#Drop the existing columns and show the updated DataFrame info
df.drop(columns=existing_drop_cols, inplace=True, errors='ignore')
df.info()

Dropping columns: ['objid', 'specobjid', 'run', 'rerun', 'camcol', 'field', 'plate', 'mjd', 'fiberid', 'redshift', 'ra', 'dec']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 31 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   u            100000 non-null  float64
 1   g            100000 non-null  float64
 2   r            100000 non-null  float64
 3   i            100000 non-null  float64
 4   z            100000 non-null  float64
 5   petroRad_u   100000 non-null  float64
 6   petroRad_g   100000 non-null  float64
 7   petroRad_i   100000 non-null  float64
 8   petroRad_r   100000 non-null  float64
 9   petroRad_z   100000 non-null  float64
 10  petroFlux_u  100000 non-null  float64
 11  petroFlux_g  100000 non-null  float64
 12  petroFlux_i  100000 non-null  float64
 13  petroFlux_r  100000 non-null  float64
 14  petroFlux_z  100000 non-null  float64
 15  petroR50_u   100000 non-nu

In [65]:
#Check for missing values in the dataset
df.isnull().sum()

u              0
g              0
r              0
i              0
z              0
petroRad_u     0
petroRad_g     0
petroRad_i     0
petroRad_r     0
petroRad_z     0
petroFlux_u    0
petroFlux_g    0
petroFlux_i    0
petroFlux_r    0
petroFlux_z    0
petroR50_u     0
petroR50_g     0
petroR50_i     0
petroR50_r     0
petroR50_z     0
psfMag_u       0
psfMag_r       0
psfMag_g       0
psfMag_i       0
psfMag_z       0
expAB_u        0
expAB_g        0
expAB_r        0
expAB_i        0
expAB_z        0
class          0
dtype: int64

In [67]:
#X contains the features (all columns except 'class'), y contains the target variable ('class')
X = df.drop(columns=['class'])
y = df['class']

#Check the unique classes in the target variable
print(df['class'].unique())

['GALAXY' 'STAR' 'QSO']


In [68]:
#Split the data into training and testing sets, then scale the features
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    #Test set will be 20% of the total data, and we set a random state for reproducibility
    test_size=0.2,
    random_state=42
)
#Scale the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Train Logistic Regression model
model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

#Make predictions on the test set
y_pred = model.predict(X_test_scaled)

#Evaluate model accuracy and performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

report = classification_report(y_test, y_pred)
print(f"Classification Report:\n{report}")

cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n{cm}")

Accuracy: 0.9653
Classification Report:
              precision    recall  f1-score   support

      GALAXY       0.99      0.98      0.99     10373
         QSO       0.88      0.84      0.86      2115
        STAR       0.96      0.97      0.97      7512

    accuracy                           0.97     20000
   macro avg       0.94      0.93      0.94     20000
weighted avg       0.97      0.97      0.97     20000

Confusion Matrix:
[[10214    80    79]
 [   85  1787   243]
 [   35   172  7305]]
